# Extension 3: ML-Based Anomaly Detection

**Goal:** Evaluate whether a KMeans distance-based approach detects more or fewer anomalies
than the existing Z-score method, by running both independently and comparing results
side-by-side.

**This notebook does NOT modify `src/transforms/performance/anomalies.py`.** It reads
the Z-score detections already written to `performance_anomalies` by Job 3, then runs
KMeans against the same raw `performance_by_version` data.

**Why KMeans for anomaly detection:**
- Handles multivariate data (p50 + p95 + p99 simultaneously)
- Does not assume a Gaussian distribution
- The 99th-percentile distance threshold adapts to the actual data shape

**Data sources:**
| Table | Role |
|---|---|
| `performance_by_version` | Raw performance metrics to cluster |
| `device_performance` | Secondary validation dataset |
| `performance_anomalies` | Existing Z-score detections for comparison |

---
**Prerequisites:** Run `make run-jobs` before opening this notebook.

## Cell 1 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import psycopg2
from pyspark.sql import SparkSession

PG = dict(
    host=os.getenv("POSTGRES_HOST", "postgres"),
    port=int(os.getenv("POSTGRES_PORT", 5432)),
    dbname=os.getenv("POSTGRES_DB", "analytics"),
    user=os.getenv("POSTGRES_USER", "analytics_user"),
    password=os.getenv("POSTGRES_PASSWORD", "analytics_pass"),
)

def pg_query(sql: str) -> pd.DataFrame:
    """Execute SQL and return a pandas DataFrame."""
    # psycopg2's context manager only manages transactions (commit/rollback),
    # NOT the connection lifecycle — close() must be called explicitly.
    conn = psycopg2.connect(**PG)
    try:
        return pd.read_sql(sql, conn)
    finally:
        conn.close()

spark = (
    SparkSession.builder
    .appName("ML Feasibility — Anomaly Detection")
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", "goodnote-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "master:", spark.sparkContext.master)

## Cell 2 — Load Performance Data

In [ ]:
perf_pd = pg_query("""
    SELECT app_version, metric_date,
           p50_duration_ms, p95_duration_ms, p99_duration_ms,
           total_interactions
    FROM performance_by_version
    ORDER BY metric_date
""")

print(f"Rows loaded: {len(perf_pd):,}")
print(f"App versions: {perf_pd['app_version'].nunique()}")
print(f"Date range  : {perf_pd['metric_date'].min()} → {perf_pd['metric_date'].max()}")
perf_pd.head()

## Cell 3 — KMeans Pipeline

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql import functions as F

PERF_COLS = ["p50_duration_ms", "p95_duration_ms", "p99_duration_ms", "total_interactions"]

# Log NaN counts before filling — a missing p50/p95/p99 is not the same as 0 ms performance.
nan_counts = perf_pd[PERF_COLS].isnull().sum()
if nan_counts.any():
    print("NaN values detected (filled with 0 before Spark ingestion):")
    print(nan_counts[nan_counts > 0].to_string())
sdf = spark.createDataFrame(perf_pd.fillna({col: 0 for col in PERF_COLS}))
sdf.cache()

assembler = VectorAssembler(inputCols=PERF_COLS, outputCol="features_raw")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True)
# k=4 chosen to capture: low-latency, mid-latency, high-latency, and high-volume performance profiles.
# Adjust if cluster sizes are highly imbalanced or silhouette scores suggest a different structure.
kmeans    = KMeans(featuresCol="features", k=4, seed=42)
pipeline  = Pipeline(stages=[assembler, scaler, kmeans])

print("Fitting KMeans pipeline (k=4)...")
model = pipeline.fit(sdf)
print("Done.")

# Collect once to avoid triggering model.transform(sdf) twice (show + collect).
preds_tmp = model.transform(sdf)
size_rows = preds_tmp.groupBy("prediction").count().orderBy("prediction").collect()
print("\nCluster sizes:")
for row in size_rows:
    print(f"  cluster {row['prediction']}: {row['count']:,}")
total   = sum(r["count"] for r in size_rows)
max_pct = max(r["count"] for r in size_rows) / total
if max_pct > 0.9:
    print(f"WARNING: Largest cluster contains {max_pct:.0%} of data. "
          "This is a degenerate solution — consider a smaller k or more feature engineering.")


### Layman Explanation

We train a KMeans clustering model on app-version performance metrics: the 50th, 95th, and 99th percentile load times, plus total interaction count. The model learns what "normal" performance looks like and groups app versions into four clusters of similar behaviour. Versions that do not fit any cluster well are the anomaly candidates we will identify next.

### Technical Discussion

The pipeline is:
```
VectorAssembler(PERF_COLS → features_raw)
  → StandardScaler(withMean=True → features)
  → KMeans(k=4, seed=42)
```

`perf_pd.fillna(0)` before `createDataFrame` ensures no NaN enters the assembler. The four PERF_COLS have very different magnitudes (p50 might be 200 ms, total_interactions might be 100,000), so scaling is essential for Euclidean-based KMeans.

`model.transform(sdf)` assigns each app-version row to a cluster (`prediction` column). The cluster sizes are printed to detect degenerate solutions — if one cluster contains 99% of rows, k=4 may be too high or the data may not have four natural groups.

### Terminology

| Term | Meaning |
|------|---------|
| **Percentile latency (p50 / p95 / p99)** | The response time below which 50%, 95%, or 99% of requests fall. p50 is the typical experience; p99 reveals tail latency that affects the slowest 1% of users. |
| **StandardScaler** | Transforms each feature to have zero mean and unit variance, preventing high-magnitude features from dominating distance calculations. |
| **KMeans (k=4)** | Divides data into exactly 4 clusters. The choice of k=4 is a design decision; there is no ground truth for "correct" k in anomaly detection. |
| **Degenerate solution** | A KMeans outcome where one cluster captures nearly all points — a sign that k is too large for the data's natural structure, or that the initialisation was poor. |
| **Feature scaling** | Rescaling input features so they live on comparable numeric ranges, preventing features with large absolute values from dominating distance-based algorithms. |

## Cell 4 — Centroid Distance Scoring

For each row, compute the Euclidean distance from its scaled feature vector to its assigned
cluster centroid. Rows whose distance exceeds the 99th percentile are flagged as anomalies.

In [ ]:
centers = model.stages[-1].clusterCenters()  # list of numpy arrays

@F.udf("double")
def centroid_dist(features, cluster_idx):
    return float(np.linalg.norm(np.array(features.toArray()) - centers[cluster_idx]))

predictions = model.transform(sdf).withColumn(
    "centroid_distance",
    centroid_dist(F.col("features"), F.col("prediction"))
)
predictions.cache()  # reused for approxQuantile, ml_anomalies, and the distance histogram

# Flag as anomaly if distance exceeds the 99th percentile
p99_dist = predictions.approxQuantile("centroid_distance", [0.99], 0.01)[0]
print(f"99th-percentile centroid distance: {p99_dist:.4f}")

ml_anomalies = predictions.filter(F.col("centroid_distance") > p99_dist)
print(f"ML anomalies detected: {ml_anomalies.count()}")

### Layman Explanation

For each app version, we measure **how far it sits from the centre of its cluster**. A version that is very close to its cluster centre looks "typical." A version far from any cluster centre looks unusual — it is an outlier. We draw the line at the top 1%: any version whose distance exceeds the 99th percentile of all distances is flagged as an anomaly.

This approach is **unsupervised** — we never had to manually label what "bad performance" looks like; the model learns the normal shape of the data and flags whatever does not fit.

### Technical Discussion

`clusterCenters()` returns a Python list of NumPy arrays — one array per cluster, each with the same dimension as `features`.

The Python UDF computes the L2 (Euclidean) norm of the vector difference between a row's feature vector and its assigned centroid:

```python
distance = ‖features − centers[cluster_idx]‖₂
```

This distance is computed in **scaled feature space** (after `StandardScaler`), so it is measured in units of standard deviations, making it comparable across features.

`approxQuantile("centroid_distance", [0.99], 0.01)` finds the 99th percentile in a single distributed pass with a maximum relative error of 1% — far more efficient than sorting the entire dataset. The resulting threshold is adaptive: it adjusts to the actual distribution of distances in the current dataset rather than using a fixed numeric cutoff.

### Terminology

| Term | Meaning |
|------|---------|
| **Euclidean distance (L2 norm)** | The straight-line distance between two points: `√(Σ(aᵢ − bᵢ)²)`. Also written `‖a − b‖₂`. |
| **Python UDF** | User Defined Function — custom Python logic registered with Spark. Each row is serialised from JVM to Python, processed, and serialised back. This is slower than native Spark operations but necessary for arbitrary Python/NumPy logic. |
| **Closure** | A function that "remembers" variables from the scope in which it was defined. Here `centroid_dist` closes over `centers` — the list of cluster centres from the trained model. |
| **approxQuantile** | A Spark method that estimates a quantile (e.g. 99th percentile) using the Greenwald-Khanna algorithm, avoiding a full sort of the dataset. The second argument (`0.01`) is the maximum relative error. |
| **Adaptive threshold** | A threshold derived from the data itself (here the 99th percentile of distances) rather than set manually. It automatically adjusts when the data distribution changes. |
| **Unsupervised anomaly detection** | Finding anomalies without labelled examples of "anomaly" vs "normal." The model infers normality from the structure of the data. |
| **Scaled feature space** | The coordinate system after applying StandardScaler, where each axis is in units of standard deviations rather than the original units (ms, counts, etc.). |

## Cell 5 — KMeans Anomaly Details

In [ ]:
ml_pd = (
    ml_anomalies
    .select("metric_date", "app_version", "centroid_distance",
            "p50_duration_ms", "p95_duration_ms", "p99_duration_ms")
    .orderBy(F.col("centroid_distance").desc())
    .toPandas()
)
print(f"KMeans anomalies (top rows):")
ml_pd.head(10)

## Cell 6 — Load Existing Z-Score Anomalies

In [ ]:
zscore_pd = pg_query("""
    SELECT metric_date, app_version, severity, z_score
    FROM performance_anomalies
    WHERE app_version IS NOT NULL
    ORDER BY metric_date
""")

print(f"Z-score anomalies in DB: {len(zscore_pd)}")
if len(zscore_pd) > 0:
    print("Severity distribution:")
    print(zscore_pd["severity"].value_counts().to_string())
zscore_pd.head()

## Cell 7 — Side-by-Side Comparison

In [ ]:
print(f"Z-score anomalies : {len(zscore_pd)}")
print(f"KMeans anomalies  : {len(ml_pd)}")

if len(zscore_pd) > 0 and len(ml_pd) > 0:
    # Normalise dates to string for merge
    zscore_pd["metric_date"] = zscore_pd["metric_date"].astype(str)
    ml_pd["metric_date"]     = ml_pd["metric_date"].astype(str)

    # Deduplicate before the inner join: performance_anomalies stores one row per
    # anomalous metric_name, so the same (metric_date, app_version) pair can appear
    # multiple times. Without dedup the join fans out and len(overlap) can exceed
    # len(zscore_pd), making the set-difference arithmetic produce negative counts.
    zscore_keys = zscore_pd[["metric_date", "app_version"]].drop_duplicates()
    ml_keys     = ml_pd[["metric_date", "app_version"]].drop_duplicates()

    overlap = pd.merge(zscore_keys, ml_keys, on=["metric_date", "app_version"])
    print(f"Detected by both  : {len(overlap)}")
    print(f"Z-score only      : {len(zscore_keys) - len(overlap)}")
    print(f"KMeans only       : {len(ml_keys) - len(overlap)}")
else:
    print("One or both anomaly sets are empty — cannot compute overlap.")


### Layman Explanation

We compare the KMeans approach with the Z-score method already running in production. This is like asking two different doctors to review the same patient records and seeing how often they agree. High overlap means both methods flag the same problems — strong evidence those really are anomalies. Cases caught by only one method are worth investigating: the Z-score-only cases may be linear outliers that KMeans misses; the KMeans-only cases may be multivariate anomalies that Z-score cannot see.

### Technical Discussion

The overlap is computed with `pd.merge(..., on=["metric_date", "app_version"])`, which is an inner join. Both DataFrames must have their dates normalised to `str` first — otherwise a `datetime` in one and an `object` in the other would silently produce zero matches.

Set arithmetic:
- `len(zscore_pd) - len(overlap)` = Z-score exclusive detections
- `len(ml_pd) - len(overlap)` = KMeans exclusive detections

This is a simplified comparison. A rigorous evaluation would also consider:
- Whether missing app_version values are handled consistently.
- Temporal proximity (anomalies on adjacent dates for the same version may be duplicates).
- Severity weighting (not all Z-score anomalies are equally serious).

### Terminology

| Term | Meaning |
|------|---------|
| **Z-score** | A measure of how many standard deviations a value is from the mean of its distribution. Values with \|z\| > 3.0 are flagged as anomalies in this project. |
| **KMeans anomaly detection** | Using cluster-distance as an anomaly score. Points far from any cluster centre are considered anomalous. |
| **Inner join (overlap)** | Keeps only rows that appear in both tables — here, app versions flagged by both methods. |
| **Set difference** | Rows in one set but not the other — here, anomalies detected by one method but missed by the other. |
| **Method comparison** | Evaluating two algorithms side by side on the same data to understand where they agree and disagree. |
| **Multivariate anomaly** | An anomaly that is only detectable when considering multiple features together. A version with p50=300ms, p95=310ms, p99=320ms might be normal on each individual metric but abnormal in their combination (no percentile spread). |

## Cell 8 — Centroid Distance Distribution

In [ ]:
import matplotlib.pyplot as plt

dist_pd = predictions.select("centroid_distance").toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(dist_pd["centroid_distance"], bins=40, edgecolor="white", color="steelblue")
ax.axvline(p99_dist, color="red", linestyle="--", label=f"p99 threshold ({p99_dist:.2f})")
ax.set_xlabel("Centroid Distance (scaled space)")
ax.set_ylabel("Count")
ax.set_title("KMeans Centroid Distance Distribution — performance_by_version")
ax.legend()
plt.tight_layout()
plt.show()

### Layman Explanation

This histogram shows how far each app-version record sits from its cluster centre. Most records cluster near zero — they are "typical." The long tail to the right contains the unusual ones. The red dashed line is our chosen threshold: anything to its right was flagged as an anomaly.

A good threshold sits at a **natural gap or inflection point** in the distribution. If the line cuts through a dense region, we are splitting "normal" records in two — an arbitrary choice. If it sits at a clear gap, the separation is meaningful.

### Technical Discussion

`predictions.select("centroid_distance").toPandas()` materialises only the one column needed for plotting, minimising the data transferred from Spark to the driver.

`ax.hist(..., bins=40)` creates 40 equal-width bins. For a right-skewed distribution (which centroid distances typically are), a log scale on the x-axis or a `bins` argument using quantile-based edges would reveal the tail structure more clearly.

`ax.axvline(p99_dist, ...)` marks the adaptive 99th-percentile threshold computed earlier. The combination of distribution + threshold line is a standard anomaly detection diagnostic — it shows both what is "normal" and where the boundary lies.

### Terminology

| Term | Meaning |
|------|---------|
| **Histogram** | A bar chart where each bar represents the count of values falling in a numeric range (bin). Used to visualise the shape of a distribution. |
| **Right-skewed distribution** | A distribution with a long tail extending to the right. Most values are low; a few are very high. Anomaly score distributions are typically right-skewed. |
| **p99 threshold** | The 99th percentile of the distance distribution. Only 1% of records exceed this value and are flagged as anomalies. |
| **Adaptive threshold** | A cut-off derived from the current data's distribution, as opposed to a hard-coded number. |
| **Distribution tail** | The low-probability region at the extremes of a distribution. Anomalies are expected to live in the tail. |
| **`toPandas()`** | Collects a Spark DataFrame from all executors back to the driver as a pandas DataFrame. Should only be used on small results; large collects can exhaust driver memory. |

## Cell 9 — Device Performance Validation

Run the same KMeans approach on `device_performance` as a cross-validation check.

In [ ]:
device_pd = pg_query("""
    SELECT device_type, metric_date,
           p50_duration_ms, p95_duration_ms, p99_duration_ms
    FROM device_performance
    ORDER BY metric_date
""")

DEVICE_COLS = ["p50_duration_ms", "p95_duration_ms", "p99_duration_ms"]
sdf_dev = spark.createDataFrame(device_pd.fillna({col: 0 for col in DEVICE_COLS}))
sdf_dev.cache()

asm_dev  = VectorAssembler(inputCols=DEVICE_COLS, outputCol="features_raw")
scl_dev  = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True)
km_dev   = KMeans(featuresCol="features", k=3, seed=42)
pipe_dev = Pipeline(stages=[asm_dev, scl_dev, km_dev])

model_dev   = pipe_dev.fit(sdf_dev)
centers_dev = model_dev.stages[-1].clusterCenters()

# Define a separate UDF capturing centers_dev, not the version-model centers.
# Reusing centroid_dist from Cell 4 would silently use wrong cluster centers.
@F.udf("double")
def centroid_dist_dev(features, cluster_idx):
    return float(np.linalg.norm(np.array(features.toArray()) - centers_dev[cluster_idx]))

preds_dev = model_dev.transform(sdf_dev).withColumn(
    "centroid_distance",
    centroid_dist_dev(F.col("features"), F.col("prediction"))
)
preds_dev.cache()  # reused for approxQuantile, count, and show

p99_dev = preds_dev.approxQuantile("centroid_distance", [0.99], 0.01)[0]
dev_anomalies = preds_dev.filter(F.col("centroid_distance") > p99_dev)
print(f"Device performance ML anomalies: {dev_anomalies.count()}")
dev_anomalies.select("device_type", "metric_date", "centroid_distance").orderBy(
    F.col("centroid_distance").desc()
).show(10)


### Layman Explanation

We repeat the same KMeans distance analysis on a **different dataset** — device performance (grouped by device type rather than app version). This is a cross-check: if the method works on app versions, does it also make sense for device types? Using a completely separate model and a fresh UDF ensures the two analyses do not accidentally share assumptions or cluster geometries.

### Technical Discussion

A fresh `Pipeline` is fit with k=3 (fewer device types than app versions justifies a smaller k). The critical fix in this cell is defining `centroid_dist_dev` as a **new UDF** that closes over `centers_dev` — the cluster centres of the device model.

The original `centroid_dist` UDF from Cell 4 closes over `centers` (the app-version model's centres). Reusing it would compute distances from each device-performance row to the **app-version centroids** — a meaningless comparison across incompatible feature spaces.

UDF closures in Spark are serialised once at definition time and broadcast to executors. `centers_dev` is captured at the moment `centroid_dist_dev` is defined, so it is guaranteed to reference the correct model regardless of execution order.

### Terminology

| Term | Meaning |
|------|---------|
| **UDF closure** | The set of variables from the outer scope that a UDF "captures" when it is defined. Changing a variable after the UDF is defined does not update what the UDF uses — the value is frozen at definition time. |
| **Cross-validation dataset** | A second, independent dataset used to test whether a method generalises beyond the dataset it was designed for. |
| **Generalisation** | The ability of a method or model to perform well on new, unseen data — not just the data it was developed on. |
| **Incompatible feature spaces** | Two datasets whose columns represent different things, making distances between their points meaningless. App-version features (4 dimensions) and device-performance features (3 dimensions) cannot share centroids. |
| **Broadcast** | Sending a read-only variable (here the centroid array) from the Spark driver to all executors so each worker can access it without repeated network transfers. |

## Cell 10 — Cleanup

In [ ]:
predictions.unpersist()
sdf.unpersist()
preds_dev.unpersist()
sdf_dev.unpersist()
spark.stop()
print("Spark session stopped.")